# Standard News Desk

Inspect this application’s recommendations, evidence grades, coverage, and research gaps. Missing analysis falls back to labelled synthetic examples. No API calls are made.

In [ ]:
from pathlib import Path
from collections import Counter
from html import escape
import json
from IPython.display import HTML, SVG, display

PROFILE = "standard"
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
path = ROOT / "analysis" / PROFILE / "latest.json"
DEMO = not path.exists()
if DEMO:
    path = ROOT / "examples" / "demo_analysis.json"
result = json.loads(path.read_text(encoding="utf-8"))
items = result["items"]
display(HTML("<strong>" + ("SYNTHETIC EXAMPLE DATA" if DEMO else escape(PROFILE.upper()) + " ANALYSIS") + "</strong>"))
print("Articles:", len(items))
print("Truncated feed texts:", sum(item.get("text_truncated", False) for item in items))
print("Source:", path)


In [ ]:
def bars(title, counts, color="#0065A9"):
    width, row, left = 760, 34, 180
    maximum = max(counts.values(), default=1) or 1
    height = 60 + row * len(counts)
    shapes = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" role="img">', f'<text x="0" y="22" font-size="18" fill="#000731">{escape(title)}</text>']
    for index, (label, value) in enumerate(counts.items()):
        y = 45 + index * row
        length = 480 * value / maximum
        shapes.extend([f'<text x="0" y="{y+16}" font-size="13">{escape(str(label))}</text>', f'<rect x="{left}" y="{y}" width="{length}" height="22" fill="{color}"/>', f'<text x="{left+length+8}" y="{y+16}" font-size="13">{value}</text>'])
    shapes.append('</svg>')
    display(SVG("".join(shapes)))

bars("Recommendation tiers", Counter(item["tier"] for item in items))
bars("Evidence grades", dict(sorted(Counter(item["source_grade"] for item in items).items())), "#A1896E")
bars("Coverage by lane", Counter(item["lane"] for item in items))


In [ ]:
headers = ["ID", "Recommendation", "Score", "Grade", "Tier", "Primary use", "Source"]
rows = []
for item in sorted(items, key=lambda item: -item["relevance"]):
    title = escape(item["headline"])
    url = item["url"]
    if url.startswith(("https://", "http://")):
        title = '<a href="' + escape(url, quote=True) + '">' + title + '</a>'
    values = [str(item["id"]), title, str(item["relevance"]), escape(item["source_grade"]), escape(item["tier"]), escape(item["primary"]), escape(item["publisher"])]
    rows.append("<tr>" + "".join("<td>" + value + "</td>" for value in values) + "</tr>")
style = "<style>table{border-collapse:collapse;width:100%;font:13px system-ui}th,td{text-align:left;padding:9px;border-bottom:1px solid #ddd}th{background:#000731;color:white}</style>"
display(HTML(style + "<table><tr>" + "".join("<th>" + h + "</th>" for h in headers) + "</tr>" + "".join(rows) + "</table>"))


In [ ]:
research = [item for item in items if item.get("research")]
fields = ["design", "sample", "population", "measure", "venue", "replication", "main_limit"]
if research:
    bars("Research records with missing evidence fields", {field.replace("_", " "): sum(not item["research"].get(field) for item in research) for field in fields}, "#381200")
else:
    print("No research-specific records in this analysis.")

usage = [entry.get("usage", {}) for entry in result.get("run", {}).get("usage", []) if not entry.get("cached")]
print("Fresh generation calls in this saved run:", len(usage))
print("Reported tokens for fresh calls:", sum(entry.get("totalTokenCount", 0) for entry in usage))
print("Token counts are usage records, not a currency estimate.")
